# South Barnegat Bay Onshore Wind Model Prediction with the Use of Long Short-Term Memory Neural Networks - LSTM Model (Depreciated)

### Written by Krupam Patel
### Written in Python 3.11.14

###### This file contains the code for the LSTM for the wind speeds to be predicted.

In [67]:
from tensorflow import keras # type: ignore
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from datetime import datetime
import os
import pandas as pd
import numpy as np

In [68]:
#Opens a new dataframe with the Clean csv
cleancsv = pd.read_csv('CLEAN.csv')

In [69]:
#Convert data into Date time and create date filter
cleancsv['Date'] = pd.to_datetime(cleancsv['Date'])
cleancsv['Date'] = cleancsv['Date'] + pd.to_timedelta(cleancsv["Hr"], unit="h")
cleancsv.drop('Hr', axis=1, inplace=True)

"""
Use this in future if data set needs specific dates
prediction = data.loc{
    (untouched_csv['Date'] > datetime(x, x, x)) &
    (untouched_csv['Date'] < datetime(x, x, x,))
}
"""

"\nUse this in future if data set needs specific dates\nprediction = data.loc{\n    (untouched_csv['Date'] > datetime(x, x, x)) &\n    (untouched_csv['Date'] < datetime(x, x, x,))\n}\n"

In [70]:
#Create month colomn and restrict to only summer months
summer_mask = cleancsv["Date"].dt.month.isin([6, 7, 8])
cleancsv = cleancsv[summer_mask].reset_index(drop=True)

In [71]:
#Prepare colomns into variables
data_main_air_temp = cleancsv['Mainland Air Temp']
data_humidity_per = cleancsv['Humidity (%)']
data_wind_direction = cleancsv['Direction (A)']
data_wind_speed = cleancsv['Wind Speed (A)']
data_gusting = cleancsv['Gusting']
data_pressure = cleancsv['Atmospheric Pressure (IN)']
data_rainfall = cleancsv['Precipitation Rate']
data_bay_temp = cleancsv['Bay Temp']
data_salinity = cleancsv['Salinity']
data_lbi_temp = cleancsv['LBI Air Temp']
data_ocean_temp = cleancsv['Ocean Temp']
data_onshore_flag = cleancsv['Onshore']
data_upwelling_flag = cleancsv['upwelling_flag']

#saves all input data into one Numpy array
dataset = np.column_stack([
    #data_main_air_temp.values,
    data_humidity_per.values,
    data_wind_direction.values,
    data_wind_speed.values,
    data_gusting.values,
    data_pressure.values,
    data_rainfall.values,
    data_bay_temp.values,
    data_salinity.values,
    data_lbi_temp.values,
    data_ocean_temp.values,
    data_onshore_flag.values,
    data_upwelling_flag.values,
])

#Save output data into variables and reshape it to be a 2d array
output_data = data_main_air_temp.values
output_data = np.array(output_data).reshape(-1, 1)

In [72]:
#Length of training data
training_data_len = int(np.ceil(len(dataset) * 0.90)) #Use 90% of training data

In [73]:
#Scaler
scaler_x= StandardScaler()
scaler_y= StandardScaler()

scaledx = scaler_x.fit_transform(dataset)
scaledy = scaler_y.fit_transform(output_data)

training_data_x = scaledx[:training_data_len] #90% of all data
training_data_y = output_data[:training_data_len] #90% of all data

X_train, y_train = [], []

In [74]:
#Sliding window over last 24 hrs
for i in range(24, training_data_len):
    X_train.append(training_data_x[i-24:i, :])
    y_train.append(training_data_y[i, 0])

#Convert lists to arrays
X_train = np.array(X_train)
y_train = np.array(y_train).reshape(-1, 1) #1D Array

In [75]:
test_x = scaledx[training_data_len-24:]
X_test = []

#rebuild window
for i in range(24, len(test_x)):
    X_test.append(test_x[i-24:i, :])

X_test = np.array(X_test)   # (samples_test, 24, n_features)
print(X_test.shape)


(218, 24, 12)


In [76]:
#Build the model
model = keras.models.Sequential()

In [77]:
#Layer Zero input_shape=(X_train.shape[1], 1)
model.add(keras.layers.Input(shape=(X_train.shape[1], X_train.shape[2])))

In [78]:
#First Layer (LSTM)
model.add(keras.layers.LSTM(32, return_sequences=False))

In [79]:
#2nd Layer (Dense)
model.add(keras.layers.Dense(32, activation="relu"))

In [80]:
#Final Output Layer (Dense)
model.add(keras.layers.Dense(1))

In [81]:
#Put all the layers together
model.summary()
model.compile(optimizer="adam",
              loss="mae",
              metrics=[keras.metrics.RootMeanSquaredError()])

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         5,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,849 (26.75 KB)

 Trainable params: 6,849 (26.75 KB)

 Non-trainable params: 0 (0.00 B)

In [82]:
#Train the model

#epochs = # of runs
#batch size = how much data is in each batch
training = model.fit(
X_train,
    y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.1,
    shuffle=True
)

Epoch 1/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 22.3171 - root_mean_squared_error: 23.0048 - val_loss: 18.2749 - val_root_mean_squared_error: 18.6374
Epoch 2/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7.9852 - root_mean_squared_error: 10.0576 - val_loss: 4.4584 - val_root_mean_squared_error: 6.6871
Epoch 3/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3.2572 - root_mean_squared_error: 4.3289 - val_loss: 4.1765 - val_root_mean_squared_error: 6.8364
Epoch 4/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.6527 - root_mean_squared_error: 3.5865 - val_loss: 5.5061 - val_root_mean_squared_error: 8.7126
Epoch 5/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.3601 - root_mean_squared_error: 3.2180 - val_loss: 3.1906 - val_root_mean_squared_error: 5.6157
Epoch 6/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.1700 - root_mean_squared_error: 2.9800 - val_loss: 1.8169 - val_root_mean_squared_error: 2.3842
Epoch 7/40
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1

In [83]:
prediction_scaled = model.predict(X_test)

# back to original units
prediction = scaler_y.inverse_transform(prediction_scaled)  


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


In [84]:
# rows of the original dataframe that correspond to X_test / prediction
test_index_start = training_data_len
test_index_end = training_data_len + prediction.shape[0]

test_df = cleancsv.iloc[test_index_start:test_index_end].copy()

# add predicted column
test_df["wind_pred"] = prediction.ravel()

#test_df.to_csv("CSV/predictions.csv", index=False)


In [85]:
# true wind speed for the same rows as predictions
y_test_true = cleancsv['Wind Speed (A)'].iloc[test_index_start:test_index_end].values

mae = mean_absolute_error(y_test_true, prediction.ravel())
print("Test MAE (m/s):", mae)

Test MAE (m/s): 134.37704561565994


In [86]:
print(prediction_scaled.shape)
print(prediction_scaled[:10].ravel())
print(prediction_scaled.mean(), prediction_scaled.std())

(218, 1)
[20.31314  20.151413 19.802576 19.214384 18.528835 17.427616 16.43124
 17.065027 24.63361  24.601055]
21.116596 7.2223635


In [87]:

training.history['val_loss']

[18.274921417236328,
 4.458431720733643,
 4.176487445831299,
 5.506092071533203,
 3.1906116008758545,
 1.8169041872024536,
 3.8945491313934326,
 2.114670753479004,
 1.9873788356781006,
 2.424241781234741,
 2.1660304069519043,
 2.3497626781463623,
 2.1408815383911133,
 2.1262571811676025,
 2.079085350036621,
 2.0544254779815674,
 1.8401731252670288,
 1.9449107646942139,
 1.896694540977478,
 2.0556106567382812,
 1.9800386428833008,
 2.0240321159362793,
 2.1069045066833496,
 2.0197131633758545,
 1.7704533338546753,
 1.846714735031128,
 1.8646830320358276,
 2.0066616535186768,
 1.9611716270446777,
 1.9122105836868286,
 1.9589380025863647,
 1.9749699831008911,
 1.9975481033325195,
 2.0261714458465576,
 2.0261456966400146,
 2.1047375202178955,
 2.2638463973999023,
 2.3294801712036133,
 2.093061923980713,
 2.247577428817749]

In [88]:
print("Wind speed mean/std:", output_data.mean(), output_data.std())
print("First 10 wind speeds:", output_data[:10].ravel())

Wind speed mean/std: 23.717628205128204 5.422786820635826
First 10 wind speeds: [13.9 13.6 12.7 11.3 10.3  9.6  9.   9.6 11.9 16.4]


In [89]:
i0 = training_data_len   # first test index
print("Example window X[0] (first test sample):")
print(cleancsv.iloc[i0-24:i0][["Wind Speed (A)", "Mainland Air Temp", "Direction (A)"]].head(10))
print("Target at i0:", cleancsv["Wind Speed (A)"].iloc[i0])

Example window X[0] (first test sample):
      Wind Speed (A)  Mainland Air Temp  Direction (A)
1942             4.2               19.6              0
1943             4.6               18.3              0
1944             4.0               17.1             15
1945             3.9               16.1             15
1946             4.3               15.2             15
1947             3.8               14.7             15
1948             4.7               14.6             15
1949             5.7               14.1             15
1950             5.7               14.3             15
1951             6.1               14.9             15
Target at i0: 3.0
